# Nested cross-validation: vit_tiny

Five outer folds evaluate grain-size classification. Three inner folds select parameters, context radius, and refit epoch count using only outer-training cells. Each cell receives one outer-held-out prediction. Separate all-data inner CV and refitting produce the deployment checkpoint. Small CNN remains outside the ensembles.

Missing-value replacement, channel statistics, and class weights use only the training portion of each fit. This is stratified spatial-group CV with same-survey context isolation, not survey-held-out evaluation. Groups connect centres at distance at most 6 for maximum radius 3. Groups stay intact, and buffer checks apply in both outer and inner folds. All folds retain all three classes. Timings include data loading during training/prediction; preprocessing and grid-search wall time are recorded separately.

**Rerun this notebook: retained historical outputs below describe the previous experiment until replaced by execution.**

## Channel-ablation setup
Select `RGB_only`, `RGB_slope`, `RGB_elevation`, or `RGB_slope_elevation` with `BEACH_CHANNEL_SETUP` (default: `RGB_slope_elevation`). Training and ensemble evaluation are independent for each setup, retaining identical splits/CV plans, parameter grids, and seeds. Only selected channels are read and normalized; the model input-channel count and saved metadata match the selection. Results are stored in the corresponding setup subfolder. Existing outputs below are retained and may predate this change; rerun to obtain setup-specific results.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    raise RuntimeError(
        "Could not locate project root containing Source and Notebooks."
    )

source_path = PROJECT_ROOT / "Source"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Source.channel_setups import get_channel_setup, setup_result_dir, validate_setup_metadata
from Source.beach_dataset import (
    find_missing_files,
    make_class_mapping,
    read_class_name_mapping,
)
from Source.models import build_vit_tiny
from Source.training import get_device
from Source.cross_validation import read_cv_plan, run_nested_cv

In [ ]:
# Select a setup here, or export BEACH_CHANNEL_SETUP before batch execution.
CHANNEL_SETUP, FEATURE_SUFFIXES = get_channel_setup()
print(f"Channel setup: {CHANNEL_SETUP}; features: {FEATURE_SUFFIXES}", flush=True)

CELL_DATA_ROOT = PROJECT_ROOT / "Data" / "Raw" / "beach_all"
CLASS_MAPPING_PATH = PROJECT_ROOT / "Data" / "Raw" / "grainsize_spec.xlsx"
PREPROCESSED_DIR = PROJECT_ROOT / "Data" / "Preprocessed"
OUTPUT_DIR = setup_result_dir(PROJECT_ROOT / "Results" / "gs_vit_tiny", CHANNEL_SETUP)

CV_DIR = PREPROCESSED_DIR

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 8
EPOCHS = 50
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 1e-4
USE_SPATIAL_AUGMENTATION = True
VIT_MODEL_NAME = "vit_tiny_patch16_224"
USE_PRETRAINED_IMAGENET_WEIGHTS = True
BASE_PARAMETER_GRID = [
    {"learning_rate": 1e-4, "weight_decay": 1e-4, "dropout": 0.0, "pretrained": USE_PRETRAINED_IMAGENET_WEIGHTS},
    {"learning_rate": 3e-5, "weight_decay": 1e-4, "dropout": 0.0, "pretrained": USE_PRETRAINED_IMAGENET_WEIGHTS},
    {"learning_rate": 1e-4, "weight_decay": 1e-3, "dropout": 0.1, "pretrained": USE_PRETRAINED_IMAGENET_WEIGHTS},
    {"learning_rate": 3e-5, "weight_decay": 1e-3, "dropout": 0.1, "pretrained": USE_PRETRAINED_IMAGENET_WEIGHTS},
]
CONTEXT_RADII = [0, 1, 2, 3]
PARAMETER_GRID = [
    {**config, "context_radius": context_radius}
    for context_radius in CONTEXT_RADII
    for config in BASE_PARAMETER_GRID
]
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)

print(f"Project root:    {PROJECT_ROOT}", flush=True)
print(f"Cell data root:  {CELL_DATA_ROOT}", flush=True)
print(f"Class mapping:   {CLASS_MAPPING_PATH}", flush=True)
print(f"Output dir:      {OUTPUT_DIR}", flush=True)
print(f"Feature order:   {FEATURE_SUFFIXES}", flush=True)

In [ ]:
print("Loading shared nested-CV plan...", flush=True)
samples, inner_folds, cv_manifest = read_cv_plan(CV_DIR)
class_to_idx = make_class_mapping(samples)
class_name_by_code = read_class_name_mapping(CLASS_MAPPING_PATH, class_to_idx.keys())
idx_to_class_code = {index: code for code, index in class_to_idx.items()}
idx_to_class_name = {
    index: f"{code}: {class_name_by_code.get(code, code)}"
    for index, code in idx_to_class_code.items()
}
print(f"Cells: {len(samples)}; outer folds: {cv_manifest['outer_folds']}; inner folds: {cv_manifest['inner_folds']}", flush=True)
display(samples.groupby(["outer_fold", "class"]).size().unstack(fill_value=0))
if max(CONTEXT_RADII) > cv_manifest["max_context_radius"]:
    raise ValueError("Regenerate the CV plan with a buffer covering every tested radius")
print(f"Independent spatial groups: {samples.spatial_group.nunique()}; centre exclusion distance: {cv_manifest['buffer_distance_cells']} cells", flush=True)

In [ ]:
if not CELL_DATA_ROOT.is_dir():
    raise FileNotFoundError(CELL_DATA_ROOT)
missing_files = find_missing_files(samples.head(20), CELL_DATA_ROOT, feature_suffixes=FEATURE_SUFFIXES)
if not missing_files.empty:
    display(missing_files)
    raise FileNotFoundError("Sampled target matrix files are missing")
print("Target/context files are loaded on demand; no full raw-data scan.", flush=True)

In [ ]:
print("Normalization will be fitted independently on each inner-training portion,", flush=True)
print("then refitted on each outer-training portion and finally on all deployment data.", flush=True)
print("NaN/-999 and unavailable context cells use that fit's training-channel means.", flush=True)

In [ ]:
print(f"Context-radius grid: {CONTEXT_RADII}", flush=True)
print(f"Raw mosaic side lengths: {[20 * (2 * radius + 1) for radius in CONTEXT_RADII]}", flush=True)
print(f"Network input size: {IMAGE_SIZE if IMAGE_SIZE is not None else 'native mosaic size'}", flush=True)

In [ ]:
device = get_device()
def model_factory(config):
    return build_vit_tiny(
        num_classes=len(class_to_idx),
        in_channels=len(FEATURE_SUFFIXES),
        pretrained=bool(config["pretrained"]),
        model_name=VIT_MODEL_NAME,
        dropout=float(config["dropout"]),
    )

print(f"Selected device: {device}; visible CUDA devices: {torch.cuda.device_count()}", flush=True)

In [ ]:
print("Starting nested cross-validation and separate deployment fitting...", flush=True)
cv_result = run_nested_cv(
    model_factory=model_factory,
    parameter_grid=PARAMETER_GRID,
    cv_dir=CV_DIR,
    cell_data_root=CELL_DATA_ROOT,
    output_dir=OUTPUT_DIR,
    feature_suffixes=FEATURE_SUFFIXES,
    channel_setup=CHANNEL_SETUP,
    model_type="vit_tiny",
    class_name_by_code=class_name_by_code,
    image_size=IMAGE_SIZE,
    model_name=VIT_MODEL_NAME,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    epochs=EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    augment=USE_SPATIAL_AUGMENTATION,
    device=device,
    random_seed=RANDOM_SEED,
)
print("Nested CV and deployment model completed.", flush=True)

In [ ]:
print("Outer-fold metrics and elapsed seconds", flush=True)
display(cv_result["fold_metrics"])
print("Mean and standard deviation across outer folds", flush=True)
display(cv_result["summary"])

In [ ]:
print("Pooled outer-held-out classification metrics: one prediction per labelled cell", flush=True)
display(cv_result["classification_report"])

In [ ]:
confusion_df = cv_result["confusion_matrix"]
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(confusion_df.to_numpy(), cmap="Blues")
ax.set_xticks(range(len(confusion_df.columns)), confusion_df.columns, rotation=45, ha="right")
ax.set_yticks(range(len(confusion_df.index)), confusion_df.index)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Pooled outer-fold predictions")
fig.colorbar(image, ax=ax)
fig.tight_layout()
plt.show()

In [ ]:
display(cv_result["oof_predictions"].head())
print("Deployment configuration (selected independently using all-data inner CV)", flush=True)
display(cv_result["deployment_configuration"])
print(f"CV artifacts: {OUTPUT_DIR / 'cross_validation'}", flush=True)
print(f"Deployment checkpoint and inference metadata: {OUTPUT_DIR}", flush=True)